**Import Libraries**

In [9]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("Libraries imported successfully")

Libraries imported successfully


**Data Generators**

In [10]:
# Image dimensions (reduced for speed)
IMG_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 5

# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation data generator
val_datagen = ImageDataGenerator(rescale=1./255)

# Load training data
train_generator = train_datagen.flow_from_directory(
    '../dataset/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# Load validation data
val_generator = val_datagen.flow_from_directory(
    '../dataset/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Class indices: {train_generator.class_indices}")
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")

Found 6043 images belonging to 2 classes.
Found 754 images belonging to 2 classes.
Training samples: 6043
Validation samples: 754
Class indices: {'with_mask': 0, 'without_mask': 1}
Image Size: 128x128
Batch Size: 16
Epochs: 5


**Model 1 — Custom CNN**

In [11]:
def create_cnn_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        
        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        
        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

cnn_model = create_cnn_model()
cnn_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,785 (432.75 KB)

 Trainable params: 110,337 (431.00 KB)

 Non-trainable params: 448 (1.75 KB)

**Model 2 — MobileNetV2**

In [12]:
def create_mobilenet_model():
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False
    
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

mobilenet_model = create_mobilenet_model()
mobilenet_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

**Model 3 — ResNet50**

In [13]:
def create_resnet50_model():
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False
    
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

resnet_model = create_resnet50_model()
resnet_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,113 (90.98 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

**Train Models**

In [14]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)

def train_model(model, model_name):
    print(f"\nTraining {model_name}...")
    history = model.fit(
        train_generator,
        steps_per_epoch=train_generator.samples // BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=val_generator,
        validation_steps=val_generator.samples // BATCH_SIZE,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    return history

print("Starting optimized training...")
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")

cnn_history = train_model(cnn_model, "Custom CNN")
mobilenet_history = train_model(mobilenet_model, "MobileNetV2")
resnet_history = train_model(resnet_model, "ResNet50")

print("\n All models trained successfully!")

Starting optimized training...
Image Size: 128x128
Batch Size: 16
Epochs: 5

Training Custom CNN...
Epoch 1/5
377/377 ━━━━━━━━━━━━━━━━━━━━ 203s 527ms/step - accuracy: 0.7350 - loss: 0.5446 - val_accuracy: 0.5492 - val_loss: 0.8056 - learning_rate: 0.0010
Epoch 2/5
377/377 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.7500 - loss: 0.5668 - val_accuracy: 0.5824 - val_loss: 0.7180 - learning_rate: 0.0010
Epoch 3/5
377/377 ━━━━━━━━━━━━━━━━━━━━ 175s 463ms/step - accuracy: 0.7732 - loss: 0.4849 - val_accuracy: 0.8404 - val_loss: 0.3757 - learning_rate: 0.0010
Epoch 4/5
377/377 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.8750 - loss: 0.3231 - val_accuracy: 0.8418 - val_loss: 0.3749 - learning_rate: 0.0010
Epoch 5/5
377/377 ━━━━━━━━━━━━━━━━━━━━ 193s 510ms/step - accuracy: 0.8450 - loss: 0.3744 - val_accuracy: 0.7553 - val_loss: 0.5484 - learning_rate: 0.0010

Training MobileNetV2...
Epoch 1/5
377/377 ━━━━━━━━━━━━━━━━━━━━ 98s 243ms/step - accuracy: 0.9509 - loss: 0.1295 - val_accuracy: 0.

 **Model Comparison**

In [17]:
import pandas as pd

results = {
    'Model': ['Custom CNN', 'MobileNetV2', 'ResNet50'],
    'Best Training Accuracy': [
        max(cnn_history.history['accuracy']),
        max(mobilenet_history.history['accuracy']),
        max(resnet_history.history['accuracy'])
    ],
    'Best Validation Accuracy': [
        max(cnn_history.history['val_accuracy']),
        max(mobilenet_history.history['val_accuracy']),
        max(resnet_history.history['val_accuracy'])
    ],
    'Best Validation Loss': [
        min(cnn_history.history['val_loss']),
        min(mobilenet_history.history['val_loss']),
        min(resnet_history.history['val_loss'])
    ]
}

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Best Validation Accuracy', ascending=False)

print("="*60)
print("MODEL COMPARISON TABLE")
print("="*60)
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
best_val_acc = results_df.iloc[0]['Best Validation Accuracy']
print(f"\nBest Performing Model: {best_model_name} (Validation Accuracy: {best_val_acc:.4f})")

results_df.to_csv('../results/model_comparison.csv', index=False)
print("\n Comparison saved to: results/model_comparison.csv")

MODEL COMPARISON TABLE
      Model  Best Training Accuracy  Best Validation Accuracy  Best Validation Loss
MobileNetV2                1.000000                  0.982713              0.041960
 Custom CNN                0.875000                  0.841755              0.374925
   ResNet50                0.589348                  0.613032              0.656219

Best Performing Model: MobileNetV2 (Validation Accuracy: 0.9827)

 Comparison saved to: results/model_comparison.csv


**Save Best Model**

In [19]:
# Select best model
if best_model_name == 'Custom CNN':
    best_model = cnn_model
elif best_model_name == 'MobileNetV2':
    best_model = mobilenet_model
else:
    best_model = resnet_model

# Save model
best_model.save('../models/face_mask_model.h5')
print(" Model saved as: models/face_mask_model.h5")

 Model saved as: models/face_mask_model.h5


In [20]:
"""
Fine-tune MobileNetV2 for better accuracy
"""

# Unfreeze some layers of MobileNetV2
base_model = mobilenet_model.layers[0]
base_model.trainable = True

# Freeze first 100 layers (keep lower layers frozen)
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with lower learning rate
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Model recompiled for fine-tuning")

# Train for 3 more epochs
history_fine = mobilenet_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=3,
    validation_data=val_generator,
    validation_steps=val_generator.samples // BATCH_SIZE,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Save the improved model
mobilenet_model.save('../models/face_mask_model_finetuned.h5')
print("✅ Fine-tuned model saved as: models/face_mask_model_finetuned.h5")

✅ Model recompiled for fine-tuning
Epoch 1/3
 15/377 ━━━━━━━━━━━━━━━━━━━━ 2:09 357ms/step - accuracy: 0.7628 - loss: 0.6566

c:\Users\RAYAN COMPUTERs\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


377/377 ━━━━━━━━━━━━━━━━━━━━ 158s 369ms/step - accuracy: 0.9139 - loss: 0.2233 - val_accuracy: 0.9880 - val_loss: 0.0336 - learning_rate: 1.0000e-05
Epoch 2/3
  1/377 ━━━━━━━━━━━━━━━━━━━━ 1:43 274ms/step - accuracy: 0.9375 - loss: 0.2253

c:\Users\RAYAN COMPUTERs\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


377/377 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - accuracy: 0.9375 - loss: 0.2253 - val_accuracy: 0.9880 - val_loss: 0.0336 - learning_rate: 1.0000e-05
Epoch 3/3
377/377 ━━━━━━━━━━━━━━━━━━━━ 140s 372ms/step - accuracy: 0.9504 - loss: 0.1372 - val_accuracy: 0.9907 - val_loss: 0.0267 - learning_rate: 1.0000e-05


✅ Fine-tuned model saved as: models/face_mask_model_finetuned.h5
